In [15]:
import geopandas as gpd
from pathlib import Path
import requests
from tqdm.auto import tqdm

In [8]:
dsm_idx_fp = "/mnt/c/Users/sebas/Documents/projects/multimodal-canopy/data/LiDAR_Data_Index/LiDAR_DSM_Index_1_2,500.shp"
dem_idx_fp = "/mnt/c/Users/sebas/Documents/projects/multimodal-canopy/data/LiDAR_Data_Index/LiDAR_DEM_Index_1_2,500.shp"
bec_fp = "/mnt/c/Users/sebas/Documents/projects/multimodal-canopy/data/bec_zones_simple.geojson"

dsm_idx_gdf = gpd.read_file(dsm_idx_fp)
dem_idx_gdf = gpd.read_file(dem_idx_fp)
bec_gdf = gpd.read_file(bec_fp)

In [9]:
print(dsm_idx_gdf.shape)
dsm_idx_gdf.head(3)

(93514, 13)


,filename,maptile,path,grid_scale,year,projection,spacing,contract,oper_name,op_number,s3Url,s3Hyperlin,geometry
0,bc_092b051_1_3_1_xli1m_utm10_20230930_20230930...,092b051_1_3_1,\092\092b\2023\dsm\bc_092b051_1_3_1_xli1m_utm1...,1:2500,2023,utm10,1 metre,OP24BMRS001,LidarBC Program,None,https://nrs.objectstore.gov.bc.ca/gdwuts/092/0...,"<a href=""https://nrs.objectstore.gov.bc.ca/gdw...","POLYGON ((1149851.39 392041.531, 1148001.886 3..."
1,bc_092b051_4_1_4_xli1m_utm10_20230930_20230930...,092b051_4_1_4,\092\092b\2023\dsm\bc_092b051_4_1_4_xli1m_utm1...,1:2500,2023,utm10,1 metre,OP24BMRS001,LidarBC Program,None,https://nrs.objectstore.gov.bc.ca/gdwuts/092/0...,"<a href=""https://nrs.objectstore.gov.bc.ca/gdw...","POLYGON ((1158972.346 396473.913, 1157124.427 ..."
2,bc_092b051_4_3_1_xli1m_utm10_20230930_20230930...,092b051_4_3_1,\092\092b\2023\dsm\bc_092b051_4_3_1_xli1m_utm1...,1:2500,2023,utm10,1 metre,OP24BMRS001,LidarBC Program,None,https://nrs.objectstore.gov.bc.ca/gdwuts/092/0...,"<a href=""https://nrs.objectstore.gov.bc.ca/gdw...","POLYGON ((1157082.755 397805.582, 1155235.259 ..."


In [10]:
print(dem_idx_gdf.shape)
dem_idx_gdf.head(3)

(93020, 13)


,filename,maptile,path,grid_scale,year,projection,spacing,contract,oper_name,op_number,s3Url,s3Hyperlin,geometry
0,bc_092b051_1_3_1_xli1m_utm10_20230930_20230930...,092b051_1_3_1,\092\092b\2023\dem\bc_092b051_1_3_1_xli1m_utm1...,1:2500,2023,utm10,1 metre,OP24BMRS001,LidarBC Program,NaN,https://nrs.objectstore.gov.bc.ca/gdwuts/092/0...,"<a href=""https://nrs.objectstore.gov.bc.ca/gdw...","POLYGON ((1149851.39 392041.531, 1148001.886 3..."
1,bc_092b051_4_1_4_xli1m_utm10_20230930_20230930...,092b051_4_1_4,\092\092b\2023\dem\bc_092b051_4_1_4_xli1m_utm1...,1:2500,2023,utm10,1 metre,OP24BMRS001,LidarBC Program,NaN,https://nrs.objectstore.gov.bc.ca/gdwuts/092/0...,"<a href=""https://nrs.objectstore.gov.bc.ca/gdw...","POLYGON ((1158972.346 396473.913, 1157124.427 ..."
2,bc_092b051_4_3_1_xli1m_utm10_20230930_20230930...,092b051_4_3_1,\092\092b\2023\dem\bc_092b051_4_3_1_xli1m_utm1...,1:2500,2023,utm10,1 metre,OP24BMRS001,LidarBC Program,NaN,https://nrs.objectstore.gov.bc.ca/gdwuts/092/0...,"<a href=""https://nrs.objectstore.gov.bc.ca/gdw...","POLYGON ((1157082.755 397805.582, 1155235.259 ..."


In [11]:
bec_gdf["ZONE"].unique()

<ArrowStringArray>
['BAFA',  'CMA',  'IMA',  'SWB',   'MH',   'PP', 'ESSF',  'IDF',  'CWH',
  'ICH',  'SBS',   'MS', 'SBPS', 'BWBS',  'CDF',   'BG']
Length: 16, dtype: str

In [16]:
sbs_gdf = bec_gdf[bec_gdf["ZONE"] == "SBS"]
sbs_gdf.head(3)

,id,FEATURE_CLASS_SKEY,ZONE,ZONE_NAME,FEATURE_AREA_SQM,FEATURE_LENGTH_M,FEATURE_AREA,FEATURE_LENGTH,OBJECTID,SE_ANNO_CAD_DATA,geometry
66,WHSE_FOREST_VEGETATION.BEC_BIOGEOCLIMATIC_ZONE...,435,SBS,Sub-Boreal Spruce,1.628717e+07,45838.0701,16096825,60057,1051,None,"POLYGON ((1299370.75 982503.562, 1300755.875 9..."
588,WHSE_FOREST_VEGETATION.BEC_BIOGEOCLIMATIC_ZONE...,435,SBS,Sub-Boreal Spruce,9.736477e+06,26623.6537,9170524,35852,1453,None,"POLYGON ((1380206.5 926774.375, 1379106.5 9269..."
783,WHSE_FOREST_VEGETATION.BEC_BIOGEOCLIMATIC_ZONE...,435,SBS,Sub-Boreal Spruce,1.433051e+07,27389.6791,13944273,29854,1661,None,"POLYGON ((1382094.73 707787.741, 1380870.625 7..."


In [29]:
sbs_gdf = sbs_gdf.to_crs(dem_idx_gdf.crs)

tiles_both_gdf = dem_idx_gdf.merge(
    dsm_idx_gdf[["maptile", "s3Url", "filename"]], 
    on="maptile", 
    how="inner", 
    suffixes=("_dem", "_dsm")
)

within_gdf = tiles_both_gdf.sjoin(
    sbs_gdf, 
    how="inner", 
    predicate="within"
)

within_gdf.sort_values(by="maptile", inplace=True)

print(within_gdf.shape)

(19145, 26)


In [33]:

dem_dir = Path("/mnt/c/Users/sebas/Documents/projects/multimodal-canopy/data/lidar_bc_dems/dtm")
dsm_dir = Path("/mnt/c/Users/sebas/Documents/projects/multimodal-canopy/data/lidar_bc_dems/dsm")

dem_dir.mkdir(parents=True, exist_ok=True)
dsm_dir.mkdir(parents=True, exist_ok=True)


def download_file(url, out_dir, filename):
    out_path = out_dir / filename

    # Don't redownload files we already have
    if out_path.exists():
        return "exists"

    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()

        with open(out_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

        return "downloaded"

    except Exception as e:
        print(f"\nFAILED: {filename}")
        print(e)

        # Remove partially downloaded file
        if out_path.exists():
            out_path.unlink()

        return "failed"

In [55]:
n_samples = 2000
step = within_gdf.shape[0] // n_samples
test_gdf = within_gdf.iloc[::step]

for row in tqdm(test_gdf.itertuples(), total=len(test_gdf)):
    download_file(row.s3Url_dem, dem_dir, row.filename_dem)
    download_file(row.s3Url_dsm, dsm_dir, row.filename_dsm)

  0%|          | 0/2128 [00:00<?, ?it/s]